In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/juliaprzygoda/Desktop/thesis-code/nft-wash-trading-detection


In [2]:
import pandas as pd
import os
import glob
from src.features.feature_pipeline import build_features
from src.detection.rule_scoring import add_wash_score
from src.detection.suspicious_transactions import (
    get_suspicious_transactions,
    calculate_suspicious_rate,
    calculate_suspicious_rate_per_collection,
)

In [3]:
opensea_preprocessed_data_folder = "data/processed/opensea/"

In [4]:
def load_parquet_folder(folder_path: str) -> pd.DataFrame:
    """
    Loads and concatenates all Parquet files from a given folder.

    Assumes folder is located relative to repository root (one level above cwd).

    Returns:
        pd.DataFrame: concatenated dataset from all Parquet files
    """

    # Resolve repository root path
    base_dir = os.path.abspath(os.path.join(os.getcwd(), "../.."))
    full_path = os.path.join(base_dir, folder_path)

    # Find parquet files
    parquet_files = glob.glob(os.path.join(full_path, "*.parquet"))

    if not parquet_files:
        print(f"⚠️ No Parquet files found in: {full_path}")
        return pd.DataFrame()

    # Load and concatenate
    dfs = [pd.read_parquet(file) for file in parquet_files]
    df = pd.concat(dfs, ignore_index=True)

    print(f"✅ Loaded {len(df)} rows from {len(parquet_files)} files")

    return df

In [5]:
# Load OpenSea NFT transaction dataset
df_raw = load_parquet_folder(opensea_preprocessed_data_folder)

# Quick preview of dataset structure
df_raw.head()

✅ Loaded 4399 rows from 3 files


,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,payment.symbol,payment.decimals,payment.token_address,token_amount,event_day
0,2025-07-28 14:43:47,1753713827,0x9f763994020190b702b13f63c16448456c225e0c9013...,ethereum,0x7d533ac6cbcee97bf00b7ca10650eab713055736,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,CryptoPunk #5418,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.170000e+19,ETH,18,0x0000000000000000000000000000000000000000,51.70,2025-07-28
1,2025-07-28 11:23:11,1753701791,0x3269f0c515ea67d593aed3c6dc50767c660a45ec6e31...,ethereum,0x799224fdb4e635f5833d66e5ba61ffcd1c1fc558,0x978852db1bf809e30576c951733227d2a8fcc07d,CryptoPunk #5512,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.733000e+19,ETH,18,0x0000000000000000000000000000000000000000,57.33,2025-07-28
2,2025-07-28 09:43:23,1753695803,0xed19e54a88fa195c9c75f22d4235a19353358ee37a8d...,ethereum,0x625202e9583039bd61cf599593d269b70521bfa1,0x24fedde1e5e2220256cb1819b7ee7f7b1b20ef5d,CryptoPunk #618,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.149000e+19,ETH,18,0x0000000000000000000000000000000000000000,51.49,2025-07-28
3,2025-07-28 08:47:47,1753692467,0x62cd1d149ad8ffbd9853a0df0b524a6a2bb2eb6cf0c4...,ethereum,0x4fc4457788f12aed1909cea1e4eeafaf2b8a4a00,0x8124ce96fbdce2cd3a80f0ba3b124a98a807be38,CryptoPunk #7364,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.850000e+19,ETH,18,0x0000000000000000000000000000000000000000,58.50,2025-07-28
4,2025-07-28 06:30:47,1753684247,0x342fc7ac6413262273136c4bba0e7a54243b32012253...,ethereum,0x86f39177283138fd6f5e344dfb78675ba4759ada,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,CryptoPunk #5512,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.450000e+19,ETH,18,0x0000000000000000000000000000000000000000,54.50,2025-07-28


In [6]:
df = df_raw.copy()

df = build_features(df)
df = add_wash_score(df)

suspicious_transactions = df[df["wash_score"] >= 40]
high_risk_transactions = df[df["wash_score"] >= 70]

print("Suspicious transactions:", len(suspicious_transactions))
print("High risk transactions:", len(high_risk_transactions))

Suspicious transactions: 18
High risk transactions: 5


In [9]:
suspicious_rate_per_collection = calculate_suspicious_rate_per_collection(
    df,
    threshold=40
)

display(suspicious_rate_per_collection)

,nft.collection,total_transactions,suspicious_transactions,suspicious_rate
0,boredapeyachtclub,1401,10,0.713776
1,cryptopunks,649,4,0.616333
2,pudgypenguins,2349,4,0.170285


In [8]:
# from pathlib import Path

# output_path = Path("../data/features/opensea_features.parquet")
# output_path.parent.mkdir(parents=True, exist_ok=True)

# df.to_parquet(output_path, index=False)